In [1]:
# Data Cleaning Notebook
# Load the raw data and prepare it for machine learning

import pandas as pd
import numpy as np

# Load the raw data (same as before)
df = pd.read_csv('../data/raw/WA_Fn-UseC_.csv')

print("Starting with raw data:")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

print("\n" + "="*60)
print("CLEANING STEP 1: Check data types and missing values")
print("="*60)

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check data types
print("\nData types:")
print(df.dtypes)


Starting with raw data:
Shape: (7043, 21)

First few rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  T

In [2]:
print("="*60)
print("CLEANING STEP 2: Fix data types and convert Yes/No to 1/0")
print("="*60)

# Fix TotalCharges: Convert from text to numeric
# (Some rows might have spaces, so errors='coerce' handles that)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check if there are any missing values after conversion
print(f"\nMissing values after TotalCharges conversion: {df['TotalCharges'].isnull().sum()}")

# If there are missing values, fill them with 0 (new customers with no charges yet)
df['TotalCharges'].fillna(0, inplace=True)

print("\n" + "-"*60)

# Convert Yes/No columns to 1/0
yes_no_columns = ['Partner', 'Dependents', 'PhoneService', 'OnlineSecurity', 
                   'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                   'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn']

for column in yes_no_columns:
    df[column] = df[column].map({'Yes': 1, 'No': 0})

print("Converted Yes/No columns to 1/0")
print(f"\nExample - Churn column now looks like:")
print(df['Churn'].head(10))

print("\n" + "-"*60)

# Remove customerID (not useful for prediction)
df = df.drop('customerID', axis=1)
print("Removed customerID column (not useful for prediction)")

print("\n" + "="*60)
print(f"Cleaned data shape: {df.shape}")
print("="*60)


CLEANING STEP 2: Fix data types and convert Yes/No to 1/0

Missing values after TotalCharges conversion: 11

------------------------------------------------------------
Converted Yes/No columns to 1/0

Example - Churn column now looks like:
0    0
1    0
2    1
3    0
4    1
5    1
6    0
7    0
8    1
9    0
Name: Churn, dtype: int64

------------------------------------------------------------
Removed customerID column (not useful for prediction)

Cleaned data shape: (7043, 20)


/var/folders/h1/njfqj7mj0_n41yb8zzq57x640000gn/T/ipykernel_81888/3069971489.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(0, inplace=True)


In [3]:
# Better way to handle the fillna (avoiding the warning)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("✅ Data cleaning complete!")
print("\n" + "="*60)
print("CLEANING STEP 3: Check cleaned data")
print("="*60)

print("\nData types after cleaning:")
print(df.dtypes)

print("\n" + "-"*60)

print("\nFirst few rows after cleaning (showing the conversion):")
print(df[['gender', 'Partner', 'Churn', 'TotalCharges']].head(10))

print("\n" + "-"*60)

print(f"\nFinal cleaned dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Check the Churn distribution one more time (should still be the same)
print("\n" + "="*60)
print("Churn distribution (should match original):")
print(df['Churn'].value_counts())

✅ Data cleaning complete!

CLEANING STEP 3: Check cleaned data

Data types after cleaning:
gender               object
SeniorCitizen         int64
Partner               int64
Dependents            int64
tenure                int64
PhoneService          int64
MultipleLines        object
InternetService      object
OnlineSecurity      float64
OnlineBackup        float64
DeviceProtection    float64
TechSupport         float64
StreamingTV         float64
StreamingMovies     float64
Contract             object
PaperlessBilling      int64
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

------------------------------------------------------------

First few rows after cleaning (showing the conversion):
   gender  Partner  Churn  TotalCharges
0  Female        1      0         29.85
1    Male        0      0       1889.50
2    Male        0      1        108.15
3    Male        0      0       1840.75
4  Female       

In [4]:
print("="*60)
print("CLEANING STEP 4: Convert remaining text columns to numbers")
print("="*60)

# For columns with just 2 categories (Male/Female), map them to 1/0
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})
print("✅ Converted gender (Male=1, Female=0)")

# For columns with 3+ categories, use OneHotEncoding
# This creates separate columns for each category (0 or 1)
categorical_columns = ['MultipleLines', 'InternetService', 'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

print(f"✅ Converted {len(categorical_columns)} categorical columns using OneHotEncoding")

print("\n" + "="*60)
print("FINAL CLEANED DATA")
print("="*60)

print(f"\nShape: {df.shape}")
print(f"All columns are now numeric (numbers only)!")

print("\nFirst few rows:")
print(df.head())

print("\n" + "="*60)
print("Next: Save this cleaned data for machine learning")
print("="*60)

# Save the cleaned data
df.to_csv('../data/cleaned_data.csv', index=False)
print("✅ Cleaned data saved to: data/cleaned_data.csv")

print(f"\nReady for modeling! Data shape: {df.shape}")
print(f"Features (columns): {df.shape[1] - 1}")  # -1 because last column is Churn
print(f"Target variable: Churn")

CLEANING STEP 4: Convert remaining text columns to numbers
✅ Converted gender (Male=1, Female=0)
✅ Converted 4 categorical columns using OneHotEncoding

FINAL CLEANED DATA

Shape: (7043, 25)
All columns are now numeric (numbers only)!

First few rows:
   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   
3       1              0        0           0      45             0   
4       0              0        0           0       2             1   

   OnlineSecurity  OnlineBackup  DeviceProtection  TechSupport  ...  Churn  \
0             0.0           1.0               0.0          0.0  ...      0   
1             1.0           0.0               1.0          0.0  ...      0   
2             1.0           1.0               0.0          0.0  ...      1   
3        